# Activity 2 — Which measurement distinguishes the groups?

**MATH4SDG / UDSM · Student investigation · 10–15 minutes plus discussion**

You receive **100 observations in two pre-existing labeled groups**, A and B, and **200 measurements** per observation. In a real application the labels might represent treatment and control or two species. These data are **synthetic classroom material**, not real biological or industrial observations.

Your task is to select a promising measurement and test whether it separates the groups in **independent fresh measurements**. The generating mechanism will be discussed later.

## 1. Inspect the supplied data

The notebook does not generate the dataset. Do not open the fresh-data file until you have written down your first conclusion.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

first = pd.read_csv("../data/exercises/feature_selection_first.csv")
features = [col for col in first.columns if col.startswith("feature_")]
assert first["sample_id"].is_unique
print("Observations:", len(first), "Measurements:", len(features))
print("Groups:", first["group"].value_counts().to_dict())
print("Missing numerical entries:", int(first[features].isna().sum().sum()))
first[["sample_id", "group"] + features[:3]].head()

## 2. Find one coordinate that appears to separate the groups

Compute the difference in sample means for every coordinate. Pick the coordinate with the **largest absolute difference**.

In [ ]:
means = first.groupby("group")[features].mean()
differences = means.loc["A"] - means.loc["B"]
chosen_feature = differences.abs().idxmax()
print("Selected measurement:", chosen_feature)
print("Mean difference (A minus B):", round(differences[chosen_feature], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot([first.loc[first["group"] == g, chosen_feature] for g in ("A", "B")])
ax.set_xticks([1, 2], ["Group A", "Group B"])
ax.set(ylabel=chosen_feature, title="Best-looking measurement in the original data")
plt.show()

A common statistical test compares two group means. But we selected this feature **after inspecting 200 measurements**. Can we interpret the resulting p-value as though we had specified the feature beforehand?

In [ ]:
a = first.loc[first["group"] == "A", chosen_feature]
b = first.loc[first["group"] == "B", chosen_feature]
nominal_p = ttest_ind(a, b, equal_var=False).pvalue
print(f"Nominal two-sided Welch-test p-value: {nominal_p:.5f}")

### Stop and record your interpretation

- If these groups represented a treatment study, what might you conclude?
- How many candidate measurements did we examine?
- How would you design a genuinely independent test?

**Our provisional conclusion:** *(write here before continuing)*

## 3. Test the very same measurement on fresh data

The new file has independent measurements for the **same labeled observation IDs**. Keep `chosen_feature` fixed. Selecting a new best feature in the fresh data would simply repeat the original search.

In [ ]:
fresh = pd.read_csv("../data/exercises/feature_selection_fresh.csv")
comparison = first[["sample_id", "group"]].merge(
    fresh, on=["sample_id", "group"], validate="one_to_one"
)
assert len(comparison) == len(first) == len(fresh)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot([comparison.loc[comparison["group"] == g, chosen_feature] for g in ("A", "B")])
ax.set_xticks([1, 2], ["Group A", "Group B"])
ax.set(ylabel=chosen_feature, title="Fresh data — SAME selected measurement")
plt.show()

In [ ]:
a_fresh = comparison.loc[comparison["group"] == "A", chosen_feature]
b_fresh = comparison.loc[comparison["group"] == "B", chosen_feature]
fresh_diff = a_fresh.mean() - b_fresh.mean()
fresh_p = ttest_ind(a_fresh, b_fresh, equal_var=False).pvalue
print(f"Original difference: {differences[chosen_feature]:.3f}")
print(f"Fresh-data difference: {fresh_diff:.3f}")
print(f"Fresh-data nominal p-value: {fresh_p:.5f}")

## 4. Adjust for the number of comparisons

One simple, conservative correction multiplies the original nominal p-value by the number of features. This is a Bonferroni adjustment, not a substitute for scientific validation.

In [ ]:
adjusted_p = min(1.0, len(features) * nominal_p)
print(f"Unadjusted p-value: {nominal_p:.5f}")
print(f"Bonferroni-adjusted p-value: {adjusted_p:.5f}")

## Bring to the group discussion

1. When does feature selection become part of the statistical test?
2. What is the difference between finding a hypothesis and testing it?
3. What should be specified **before collecting data** in a biological or industrial experiment?
4. How might an exploratory analysis still be valuable when its most impressive result fails to replicate?

**The facilitator will reveal the construction of the synthetic measurements after the activity.**